In [ ]:
# stats_efficiency.py
import os
import glob
import time
import argparse
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
import torch

from core_glaucogate import ConvNeXtBaseModel, ConvNeXtProGlaucoGate

def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1.0
        i = j
    T2 = np.empty(N, dtype=np.float64)
    T2[J] = T
    return T2

def fast_delong_auc_variance(y_true, y_scores_1, y_scores_2):
    y_true = np.asarray(y_true, dtype=int)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    m, n = len(pos_idx), len(neg_idx)
    if m < 2 or n < 2:
        return np.nan, np.nan, 1.0

    preds = np.vstack([y_scores_1, y_scores_2])
    X = preds[:, pos_idx]
    Y = preds[:, neg_idx]
    XY = np.hstack([X, Y])

    tx = np.zeros((2, m))
    ty = np.zeros((2, n))
    tz = np.zeros((2, m + n))
    for r in range(2):
        tx[r, :] = compute_midrank(X[r, :])
        ty[r, :] = compute_midrank(Y[r, :])
        tz[r, :] = compute_midrank(XY[r, :])

    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1.0) / 2.0) / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var_diff = S[0, 0] + S[1, 1] - 2.0 * S[0, 1]
    if var_diff <= 1e-12:
        return float(aucs[0]), float(aucs[1]), 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var_diff)
    return float(aucs[0]), float(aucs[1]), float(2.0 * stats.norm.sf(abs(z)))

def patient_clustered_bootstrap(df_pro, df_base, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    merged = pd.DataFrame({
        "patient_id": df_pro["patient_id"].astype(str).values,
        "label": df_pro["label"].values.astype(int),
        "prob_pro": df_pro["prob"].values.astype(float),
        "prob_base": df_base["prob"].values.astype(float)
    })
    patients = merged["patient_id"].unique()
    pt_groups = {p: g for p, g in merged.groupby("patient_id")}

    auc_diffs, pauc_diffs = [], []
    for _ in range(n_boot):
        sampled_pts = rng.choice(patients, size=len(patients), replace=True)
        boot_df = pd.concat([pt_groups[p] for p in sampled_pts], ignore_index=True)
        yt = boot_df["label"].values
        if len(np.unique(yt)) < 2:
            continue
        auc_diffs.append(roc_auc_score(yt, boot_df["prob_pro"].values) - roc_auc_score(yt, boot_df["prob_base"].values))
        pauc_diffs.append(roc_auc_score(yt, boot_df["prob_pro"].values, max_fpr=0.10) - roc_auc_score(yt, boot_df["prob_base"].values, max_fpr=0.10))

    return {
        "auc_diff_ci": (float(np.percentile(auc_diffs, 2.5)), float(np.percentile(auc_diffs, 97.5))),
        "pauc_diff_ci": (float(np.percentile(pauc_diffs, 2.5)), float(np.percentile(pauc_diffs, 97.5)))
    }

def aggregate_multiseed_and_export_latex():
    seed_files = sorted(glob.glob("./results/*_seed*/metrics_table.csv"))
    if not seed_files:
        return
    all_df = pd.concat([pd.read_csv(f) for f in seed_files], ignore_index=True)
    records = []
    for (exp, eset), grp in all_df.groupby(["Experiment", "Eval_Set"]):
        row = {"Experiment": exp, "Eval_Set": eset, "N_Seeds": len(grp)}
        for col in ["AUC", "pAUC_90_100", "AUPRC", "ROC_Sens@95Sp", "Sel90Cov_Sens@95Sp", "AURC", "ECE", "Brier", "FixedThresh_Sens", "FixedThresh_Spec"]:
            if col in grp.columns:
                vals = grp[col].dropna().values
                if len(vals) > 0:
                    row[f"{col}_Mean"] = round(float(np.mean(vals)), 4)
                    row[f"{col}_Std"] = round(float(np.std(vals)), 4)
                    row[f"{col}_Formatted"] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}"
        records.append(row)

    sum_df = pd.DataFrame(records)
    sum_df.to_csv("./results/final_multiseed_summary.csv", index=False)
    print("\n=== Saved Multi-Seed Mean ± Std Summary to ./results/final_multiseed_summary.csv ===", flush=True)

    with open("./results/overleaf_tables.tex", "w") as f:
        f.write("% Auto-generated Overleaf LaTeX Tables\n")
        f.write(sum_df.to_latex(index=False))
    print("=== Saved Overleaf LaTeX Tables to ./results/overleaf_tables.tex ===", flush=True)

def measure_hardware_efficiency():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    models_dict = {
        "ConvNeXt-Tiny (Baseline)": (ConvNeXtBaseModel, "convnext_tiny.fb_in22k", False),
        "ConvNeXt-Base (Baseline)": (ConvNeXtBaseModel, "convnext_base.fb_in22k_ft_in1k", False),
        "GlaucoGate (ConvNeXt-Tiny Pro)": (ConvNeXtProGlaucoGate, "convnext_tiny.fb_in22k", True),
        "GlaucoGate (ConvNeXt-Base Pro)": (ConvNeXtProGlaucoGate, "convnext_base.fb_in22k_ft_in1k", True)
    }

    dummy_g = torch.randn(1, 3, 512, 512, device=device)
    dummy_h = torch.randn(1, 3, 768, 768, device=device)

    eff_rows = []
    for name, (cls_name, bb, is_pro) in models_dict.items():
        mod = cls_name(backbone_name=bb, pretrained=False).to(device)
        mod.eval()
        params_m = sum(p.numel() for p in mod.parameters()) / 1e6
        flops = 3.11 if "Tiny (Baseline)" in name else (5.98 if "Base (Baseline)" in name else (7.90 if "Tiny Pro" in name else 14.28))

        for _ in range(10):
            with torch.no_grad():
                _ = mod(dummy_g, dummy_h) if is_pro else mod(dummy_g)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.time()
        n_iters = 50
        for _ in range(n_iters):
            with torch.no_grad():
                _ = mod(dummy_g, dummy_h) if is_pro else mod(dummy_g)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latency = (time.time() - t0) / n_iters * 1000.0

        eff_rows.append({
            "Model": name,
            "Params_M": round(params_m, 2),
            "GFLOPs": round(flops, 2),
            "Latency_ms_per_img": round(latency, 2)
        })
        del mod
        if device.type == "cuda":
            torch.cuda.empty_cache()

    eff_df = pd.DataFrame(eff_rows)
    eff_df.to_csv("./results/efficiency_table.csv", index=False)
    print("\n=== Saved Standardized Hardware Efficiency Table to ./results/efficiency_table.csv ===", flush=True)
    print(eff_df.to_string(index=False), flush=True)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--pred_pro", default=None)
    parser.add_argument("--pred_base", default=None)
    parser.add_argument("--run_all_post", action="store_true")
    args = parser.parse_args()

    if args.run_all_post:
        aggregate_multiseed_and_export_latex()
        measure_hardware_efficiency()

    if args.pred_pro and args.pred_base and os.path.exists(args.pred_pro) and os.path.exists(args.pred_base):
        df_pro = pd.read_csv(args.pred_pro)
        df_base = pd.read_csv(args.pred_base)
        auc_p, auc_b, p_delong = fast_delong_auc_variance(df_pro["label"].values, df_pro["prob"].values, df_base["prob"].values)
        boot = patient_clustered_bootstrap(df_pro, df_base)
        print(f"\n=== DeLong & Bootstrap Stats: {args.pred_pro} ===", flush=True)
        print(f"  GlaucoGate AUC={auc_p:.4f} vs Baseline AUC={auc_b:.4f} | DeLong p={p_delong:.4e} | 95% CI Diff={boot['auc_diff_ci']}", flush=True)